In [1]:
pip install python-dotenv

Note: you may need to restart the kernel to use updated packages.


In [2]:
import pandas as pd

In [3]:
import os
from dotenv import load_dotenv
from sqlalchemy import create_engine

In [4]:
load_dotenv()

True

In [5]:
db_host = os.getenv("DB_HOST")
db_user = os.getenv("DB_user")
db_pass = os.getenv("DB_PASSWORD")
db_name = os.getenv("DB_NAME")

In [6]:
engine = create_engine(f"mysql+mysqlconnector://{db_user}:{db_pass}@{db_host}/{db_name}")

# 1.Financial KPIs

*Business Goal:* To provide the finance department with an immediate overview of current asset valuation, total units on hand, and the total financial incentives (savings) offered to customers.

In [7]:
query = """
SELECT 
    COUNT(DISTINCT name) AS Total_Unique_Products,      
    SUM(availableQuantity) AS Total_Items_In_Stock,      
    SUM(total_stock_value) AS Total_Inventory_Value,   
    SUM(money_saved * availableQuantity) AS Total_Consumer_Savings  
FROM product_inventory;"""
result = pd.read_sql(query, engine)
result


,Total_Unique_Products,Total_Items_In_Stock,Total_Inventory_Value,Total_Consumer_Savings
0,1676,14960.0,224939407.0,24338693.0


# 2.Category Performance Analysis

*Business Goal:* To identify which product categories hold the highest capital concentration in the warehouse and assess which categories are driving promotional strategies through discount rates.

In [8]:
query = """
SELECT 
    Category,
    COUNT(name) AS Product_Count,
    SUM(availableQuantity) AS Total_Quantity,
    ROUND(SUM(total_stock_value), 2) AS Category_Stock_Value,
    ROUND(AVG(discountPercent), 2) AS Average_Discount_Rate
FROM product_inventory
GROUP BY Category
ORDER BY Category_Stock_Value DESC;
"""

In [9]:
result = pd.read_sql(query, engine)
result

,Category,Product_Count,Total_Quantity,Category_Stock_Value,Average_Discount_Rate
0,Cooking Essentials,514,2186.0,33835972.0,7.16
1,Munchies,514,2186.0,33835972.0,7.16
2,Personal Care,344,1458.0,27151848.0,6.25
3,Paan Corner,344,1458.0,27151848.0,6.25
4,Packaged Food,388,1521.0,22502620.0,8.32
5,Ice Cream & Desserts,388,1521.0,22502620.0,8.32
6,Chocolates & Candies,388,1521.0,22502620.0,8.32
7,Home & Cleaning,194,839.0,12300461.0,5.68
8,Health & Hygiene,97,425.0,6432274.0,8.05
9,"Dairy, Bread & Batter",129,485.0,5523273.0,7.16


# 3.Out of Stock & Risk Analysis

*Business Goal:* An operational query designed to alert procurement and supply chain teams about stockouts, helping them mitigate supply chain risks and calculate stockout ratios across categories.

In [10]:
query = """
SELECT 
    Category,
    COUNT(CASE WHEN outOfStock = 1 THEN 1 END) AS Out_Of_Stock_Count, 
    COUNT(name) AS Total_Category_Products,
    ROUND((COUNT(CASE WHEN outOfStock = 1 THEN 1 END) / COUNT(name)) * 100, 2) AS Out_Of_Stock_Percentage
FROM product_inventory
GROUP BY Category
ORDER BY Out_Of_Stock_Percentage DESC;"""
result = pd.read_sql(query, engine)
result

,Category,Out_Of_Stock_Count,Total_Category_Products,Out_Of_Stock_Percentage
0,Biscuits,42,147,28.57
1,"Dairy, Bread & Batter",28,129,21.71
2,Beverages,28,129,21.71
3,"Meats, Fish & Eggs",12,63,19.05
4,Health & Hygiene,13,97,13.40
5,Cooking Essentials,64,514,12.45
6,Munchies,64,514,12.45
7,Packaged Food,45,388,11.60
8,Ice Cream & Desserts,45,388,11.60
9,Chocolates & Candies,45,388,11.60


# 4.Top 5 High-Value Products

> *Business Goal:* To pinpoint the top 5 flagship products that tie up the largest share of working capital, helping inventory managers optimize storage and turnover strategies for high-exposure items.
> 


In [11]:
query = """
SELECT 
    name,
    Category,
    availableQuantity,
    discountedSellingPrice,
    total_stock_value
FROM product_inventory
ORDER BY total_stock_value DESC
LIMIT 5;"""
result=pd.read_sql(query,engine)
result

,name,Category,availableQuantity,discountedSellingPrice,total_stock_value
0,Borges Extra Light Olive Oil Bottle,Munchies,6,140400.0,842400.0
1,Borges Extra Light Olive Oil Bottle,Cooking Essentials,6,140400.0,842400.0
2,Praakritik Natural Desi Gir Cow A2 Ghee,Cooking Essentials,6,130500.0,783000.0
3,Praakritik Natural Desi Gir Cow A2 Ghee,Munchies,6,130500.0,783000.0
4,Saffola Gold (Jar),Munchies,6,124000.0,744000.0
